<a href="https://colab.research.google.com/github/eddieUlalan/bco7006/blob/main/assessment4_case3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Assessment 4 Case 3**

#Reading the Data

In [ ]:
import pandas as pd
import numpy as np

# from subprocess import check_output
# print(check_output(["ls", "../input"]).decode("utf8"))

train = pd.read_csv("https://raw.githubusercontent.com/eddieUlalan/bco7006/refs/heads/main/labeledTrainData.tsv",
                    header=0,
                    delimiter="\t",
                    quoting=3)

print("The shape of our data:", train.shape)
print("Our column names are:", train.columns.values)

##Explanation

**What it does**

Loads the IMDB movie review dataset into a pandas DataFrame, checks that the input files are available, and prints the dataset size and column structure.

**Libraries used**

*   pandas (imported as pd) - used to load and work with the dataset as a DataFrame.
*   numpy (imported as np) - imported for numerical calculations.
*   check_output (imported from subprocess) - used to display the files available in the input directory.

**pandas / numpy methods:**

*   pd.read_csv() - reads the number of rows and columns in the dataset.
*   train.shape - returns the number of rows and columns in the dataset.
*   train.columns.values - returns the names of all columns in the dataset.

**Why it matters for the business**

This step ensures the dataset is correctly loaded and structured before any analysis begins. By confirming the number of rows and columns, we verify that all reviews are available and ready for processing, which prevents errors later in the analysis pipeline.

#Data Cleaning and Text Preprocessing


In [ ]:
from bs4 import BeautifulSoup
import re
import nltk
from nltk.corpus import stopwords
# import nltk
# nltk.download('stopwords')

example1 = BeautifulSoup(train["review"][0])

letters_only = re.sub("[^a-zA-Z]", " ", example1.get_text())

lower_case = letters_only.lower()
words = lower_case.split()

words = [w for w in words if not w in stopwords.words("english")]

def review_to_words(raw_review):
    review_text = BeautifulSoup(raw_review).get_text()
    letters_only = re.sub("[^a-zA-Z]", " ", review_text)
    words = letters_only.lower().split()
    stops = set(stopwords.words("english"))
    meaningful_words = [w for w in words if not w in stops]
    return " ".join(meaningful_words)

clean_review = review_to_words( train["review"][0] )

num_reviews = train["review"].size

clean_train_reviews = []
for i in range(len(train["review"])):
    clean_train_reviews.append(review_to_words(train["review"][i]))

##Explanation

**What is does**: Cleans raw movie reviews by removing HTML tags, non-alphabetical characters, conversion to lowercase letters, removal of stop words, and this data cleaning technique is applied to all reviews in the dataset.

**Libraries used**:

- BeautifulSoup (bs4) — removes HTML tags from raw review text.

- re (regular expressions) — removes non-alphabet characters from text.

- nltk.corpus.stopwords — provides a list of common English stopwords to remove uninformative words.

**pandas / Python methods**

- train["review"] — selects the review column from the DataFrame.
- BeautifulSoup(...).get_text() — extracts plain text from HTML content.
- re.sub("[^a-zA-Z]", " ", text) — replaces all non-letter characters with spaces.
- .lower() — converts text to lowercase.
- .split() — splits text into individual words.
stopwords.words("english") — returns a list of English stopwords.

**Loops / conditions**
- for i in range(len(train["review"])): — loops through every review in the dataset and applies the cleaning function.
- [w for w in words if not w in stopwords.words("english")] — removes stopwords using a list comprehension (filters words conditionally).

**Why it matters for the business**

This step converts messy, unstructured movie reviews into clean and consistent text data. Removing HTML tags, punctuation, and common stopwords ensures that only meaningful words remain, which improves the quality of any analysis or prediction made from the data.

#Creating Features from Bag of Words + Model Training

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(analyzer="word",
                             tokenizer=None,
                             preprocessor=None,
                             stop_words=None,
                             max_features=5000)

train_data_features = vectorizer.fit_transform(clean_train_reviews)
train_data_features = train_data_features.toarray()

print( "Training the random forest...")
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators = 100)

forest = forest.fit( train_data_features, train["sentiment"] )

#Explanation

**What it does**

Converts the cleaned movie reviews into numeric features using a Bag of Words representation and trains a Random Forest model to predict review sentiment(positive or negative).

**Libraries used**


*   CountVectorizer (imported from sklearn.feature_extraction.text) - converts text reviews into numerical feature vectors using word frequency counts.
*   RandomForestClassifier (imported from sklearn.ensemble) - used to build a machine learning model that classifies reviews as positive or negative.

**Loop / conditions**




**scikit-learn methods:**


*   CountVectorizer(...) — creates a Bag of Words vectorizer.
*   max_features=5000 — keeps only the 5,000 most frequent words in the dataset.
*   fit_transform(clean_train_reviews) — learns the vocabulary from the reviews and converts the reviews into a document-term matrix.
*   toarray() — converts the sparse matrix into a standard NumPy array.
*   RandomForestClassifier(n_estimators=100) — creates a Random Forest model containing 100 decision trees.
*   fit(train_data_features, train["sentiment"]) — trains the model using the review features as inputs and the sentiment labels as target values.

**Why it matters for the business**

This step transforms cleaned text into numerical data that can be understood by machine learning models and trains a system that can automatically classify movie reviews as positive or negative. This enables large-scale sentiment analysis without manual reading of each review.


# Code Improvements

In [ ]:
# ── COMPARISON: Original vs Improved Text Cleaning ──────────────────────────

# --- ORIGINAL (unchanged from above) ----------------------------------------
# def review_to_words(raw_review):
#     review_text = BeautifulSoup(raw_review).get_text()   # Remove HTML
#     letters_only = re.sub("[^a-zA-Z]", " ", review_text) # Remove non-letters
#     words = letters_only.lower().split()                   # Lowercase + split
#     stops = set(stopwords.words("english"))               # Load stopwords
#     meaningful_words = [w for w in words if not w in stops] # Filter stopwords
#     return " ".join(meaningful_words)
#
# LIMITATION: No stemming → "running", "runs", "ran" are THREE separate features

# --- IMPROVED: Adds Porter Stemmer -------------------------------------------
import nltk
nltk.download('stopwords', quiet=True)
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
import re

stemmer = PorterStemmer()   # Initialise the stemmer once (efficient)

def review_to_words_improved(raw_review):
    """
    Improved version of review_to_words().
    Adds Porter Stemming as a final step to reduce words to their root form.
    All other steps are identical to the original function.
    """
    # Step 1: Remove HTML (same as original)
    review_text = BeautifulSoup(raw_review, "html.parser").get_text()
    # Step 2: Remove non-letters (same as original)
    letters_only = re.sub("[^a-zA-Z]", " ", review_text)
    # Step 3: Lowercase and split (same as original)
    words = letters_only.lower().split()
    # Step 4: Remove stopwords (same as original)
    stops = set(stopwords.words("english"))
    meaningful_words = [w for w in words if not w in stops]
    # Step 5: NEW — Apply Porter Stemmer to each word
    #   "running" → "run", "runs" → "run", "ran" → "ran" (reduced to root form)
    stemmed_words = [stemmer.stem(w) for w in meaningful_words]
    return " ".join(stemmed_words)

# Show the difference on one review
example_review = train["review"][0]
original_clean  = review_to_words(example_review)
improved_clean  = review_to_words_improved(example_review)

print("=" * 60)
print("ORIGINAL cleaned (first 200 chars):")
print(original_clean[:200])
print()
print("IMPROVED cleaned with stemming (first 200 chars):")
print(improved_clean[:200])
print("=" * 60)

# Apply improved cleaning to all training reviews
print("\nApplying improved cleaning to all 25,000 reviews...")
clean_train_reviews_improved = []
for i in range(len(train["review"])):
    clean_train_reviews_improved.append(review_to_words_improved(train["review"][i]))
print(f"Done. {len(clean_train_reviews_improved)} reviews cleaned with stemming.")

##Explanation

**What it does**

Defines an improved version of the original cleaning function, `review_to_words_improved()`, which adds Porter Stemming as a final step to reduce each word to its root form. Applies the improved function to all 25,000 reviews and stores the cleaned results in `clean_train_reviews_improved`.

**Libraries used**

*   `nltk.stem.PorterStemmer` — reduces words to their root form using the Porter stemming algorithm. For example: "running" → "run", "runs" → "run", "lovely" → "love". Initialised once before the loop to avoid recreating the object 25,000 times.
*   `BeautifulSoup` (bs4) — removes HTML tags from raw review text (same as original).
*   `re` (regular expressions) — removes non-alphabet characters from text (same as original).
*   `nltk.corpus.stopwords` — provides the list of common English stopwords to filter out (same as original).

**pandas / numpy methods:**

*   `train["review"][i]` — selects a single review string from the review column by row index.
*   `nltk.download("stopwords", quiet=True)` — downloads the stopwords data file if not already cached; `quiet=True` suppresses the progress message.
*   `BeautifulSoup(raw_review, "html.parser").get_text()` — strips all HTML tags and returns plain text.
*   `re.sub("[^a-zA-Z]", " ", review_text)` — replaces every non-letter character with a space.
*   `" ".join(stemmed_words)` — reassembles the stemmed word list into a single string.

**Loops / conditions**

*   `[w for w in words if not w in stops]` — list comprehension that conditionally keeps only words not in the stopword set (same as original).
*   `[stemmer.stem(w) for w in meaningful_words]` — **NEW**: list comprehension that applies the Porter Stemmer to every word that survived stopword removal, converting each to its root form.
*   `for i in range(len(train["review"])):` — loops through all 25,000 reviews and appends the improved cleaned string to `clean_train_reviews_improved`.

**Why it matters for the business**

Without stemming, different grammatical forms of the same word — such as "running", "runs", and "ran" — are treated as three entirely separate features even though they represent the same concept. This unnecessarily inflates the vocabulary size and can reduce model efficiency. Stemming consolidates these forms into a single feature, reducing noise and helping the model generalise better across the variety of language used in customer reviews.

In [ ]:
# ── IMPROVED METHOD: TF-IDF + Bigrams + Logistic Regression + Cross-Validation ─

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

print("=" * 65)
print(" ORIGINAL METHOD: CountVectorizer + Random Forest")
print("=" * 65)

# --- ORIGINAL (recap): already fitted as 'vectorizer' and 'forest' above ----
# CountVectorizer: max_features=5000, unigrams only
# RandomForestClassifier: n_estimators=100
# No validation split — trains on all data, no test accuracy reported

# Run cross-validation on the ORIGINAL features so we have a fair baseline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer

vec_orig = CountVectorizer(analyzer="word", max_features=5000)
X_orig   = vec_orig.fit_transform(clean_train_reviews).toarray()
y        = train["sentiment"]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_orig = cross_val_score(
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    X_orig, y, cv=cv, scoring="accuracy"
)
print(f"Cross-val accuracy per fold : {[f'{s:.4f}' for s in scores_orig]}")
print(f"Mean accuracy               : {scores_orig.mean():.4f}")
print(f"Standard deviation          : {scores_orig.std():.4f}")

print()
print("=" * 65)
print(" IMPROVED METHOD: TF-IDF (+ bigrams) + Logistic Regression")
print("=" * 65)

# IMPROVEMENT 1 & 2: TF-IDF with unigrams AND bigrams
# - TF-IDF weights rare, discriminative words higher than common words
# - ngram_range=(1,2) adds two-word phrase features (bigrams)
# - min_df=2 ignores words appearing in fewer than 2 reviews
# - sublinear_tf=True applies log-scaling to term frequencies
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),    # IMPROVEMENT 2: include bigrams
    min_df=2,
    sublinear_tf=True      # IMPROVEMENT 1: log-scaled TF-IDF weighting
)

# Use improved cleaned reviews (with stemming) produced in previous section
X_improved = tfidf_vectorizer.fit_transform(clean_train_reviews_improved)

# Logistic Regression: interpretable, fast, strong baseline for text tasks
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# IMPROVEMENT 4: 5-fold stratified cross-validation
# - Stratified: each fold preserves the original class ratio (50/50)
# - 5 folds: train on 4, test on 1, rotate — reports true generalisation
scores_improved = cross_val_score(
    lr_model, X_improved, y,
    cv=cv, scoring="accuracy"
)
print(f"Cross-val accuracy per fold : {[f'{s:.4f}' for s in scores_improved]}")
print(f"Mean accuracy               : {scores_improved.mean():.4f}")
print(f"Standard deviation          : {scores_improved.std():.4f}")

# ── Side-by-side result summary ──────────────────────────────────────────────
print()
print("=" * 65)
print(" RESULTS COMPARISON (5-fold cross-validation)")
print("=" * 65)
gain = (scores_improved.mean() - scores_orig.mean()) * 100
print(f"{'Method':<40} {'Mean Acc':>9} {'Std Dev':>9}")
print("-" * 65)
print(f"{'Original  (BoW + Random Forest)':<40} {scores_orig.mean():>9.4f} {scores_orig.std():>9.4f}")
print(f"{'Improved  (TF-IDF + Logistic Regression)':<40} {scores_improved.mean():>9.4f} {scores_improved.std():>9.4f}")
print("-" * 65)
print(f"Accuracy gain from improvements: {gain:+.2f} percentage points")
print("=" * 65)

# ── Feature interpretability: top words driving each class ───────────────────
lr_model.fit(X_improved, y)   # Fit on full dataset for feature inspection
feature_names = tfidf_vectorizer.get_feature_names_out()
coef          = lr_model.coef_[0]

top_pos = [feature_names[i] for i in coef.argsort()[-10:][::-1]]
top_neg = [feature_names[i] for i in coef.argsort()[:10]]

print("\nTop 10 features predicting POSITIVE sentiment:", top_pos)
print("Top 10 features predicting NEGATIVE sentiment:", top_neg)
print("\n(These weights are unique to Logistic Regression — Random Forest")
print(" cannot produce this kind of interpretable per-word explanation.)")

##Explanation

**What it does**

Implements four improvements to the original pipeline: re-vectorises the cleaned reviews using TF-IDF with bigram phrases, trains a Logistic Regression classifier, evaluates both the original and improved models using 5-fold cross-validation, and extracts the most predictive words and phrases using coefficient weights for direct comparison.

**Libraries used**

*   `TfidfVectorizer` (from `sklearn.feature_extraction.text`) — replaces `CountVectorizer`; weights each word by how distinctive it is across the dataset, and supports bigrams via `ngram_range=(1,2)`.
*   `LogisticRegression` (from `sklearn.linear_model`) — replaces `RandomForestClassifier`; a fast, interpretable classifier that assigns a weight to each feature, showing exactly which words drive positive or negative predictions.
*   `StratifiedKFold` (from `sklearn.model_selection`) — splits the dataset into 5 folds while preserving the 50/50 class ratio in each fold.
*   `cross_val_score` (from `sklearn.model_selection`) — evaluates a model across all 5 folds and returns one accuracy score per fold.
*   `numpy` (imported as `np`) — used for array operations including `argsort()` to identify the highest and lowest coefficient features.

**pandas / numpy methods:**

*   `TfidfVectorizer(ngram_range=(1,2), max_features=10000, min_df=2, sublinear_tf=True)` — `ngram_range=(1,2)` adds two-word phrase features alongside single words; `min_df=2` ignores terms in fewer than 2 reviews; `sublinear_tf=True` applies log scaling to term frequencies to reduce the dominance of very common words.
*   `tfidf_vectorizer.fit_transform(clean_train_reviews_improved)` — builds the TF-IDF vocabulary and transforms all reviews into weighted feature vectors in one step.
*   `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` — `shuffle=True` randomises fold assignment; `random_state=42` fixes the shuffle for reproducibility.
*   `cross_val_score(..., scoring="accuracy")` — returns a numpy array of 5 accuracy values; `.mean()` and `.std()` report average performance and consistency across folds.
*   `lr_model.coef_[0]` — after fitting, retrieves the 1D array of learned weights (one per feature); `coef_` is 2D so `[0]` extracts the single row.
*   `coef.argsort()` — numpy method returning the indices that would sort the array in ascending order; `[:10]` selects the 10 lowest (most negative) and `[-10:][::-1]` selects the 10 highest (most positive) in descending order.
*   `tfidf_vectorizer.get_feature_names_out()` — returns all vocabulary strings in index order so `feature_names[i]` gives the word or bigram phrase at position `i`.

**Loops / conditions**

*   `[feature_names[i] for i in coef.argsort()[-10:][::-1]]` — list comprehension iterating over the 10 highest-coefficient indices to retrieve the top positive feature names.
*   `[feature_names[i] for i in coef.argsort()[:10]]` — list comprehension iterating over the 10 lowest-coefficient indices to retrieve the top negative feature names.

**Why it matters for the business**

TF-IDF ensures that words appearing in almost every review — such as "film" or "movie" — do not dominate the model, allowing genuinely informative words to carry more weight. Bigrams allow the model to detect phrases such as "not good" or "highly recommend" whose meaning cannot be captured by individual words alone. Logistic Regression provides interpretable feature weights, enabling business users to inspect and validate exactly which words and phrases are driving positive or negative predictions. Cross-validation delivers a reliable performance estimate that is not dependent on a single lucky or unlucky data split.